<h1>Q1</h1>

In [5]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import re
import time
from collections import Counter
from datasets import load_dataset
from snorkel.labeling import labeling_function, PandasLFApplier
from snorkel.labeling import LFAnalysis
from snorkel.labeling.model import MajorityLabelVoter

# wandb.login()

# Define global constants for Snorkel
ABSTAIN = -1
# CoNLL-2003 tag indices (9 total classes: 0-8)
O, B_PER, I_PER, B_LOC, I_LOC, B_ORG, I_ORG, B_MISC, I_MISC = range(9)
LABEL_NAMES = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-MISC', 'I-MISC']
CARDINALITY = 9


# Load the dataset
dataset = load_dataset("eriktks/conll2003")
train_data = dataset["train"]

# Calculate Statistics
num_train = len(train_data)
num_val = len(dataset["validation"])
num_test = len(dataset["test"])
TARGET_ENTITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8] 

all_ner_tags = []
for item in train_data:
    all_ner_tags.extend(item['ner_tags'])

entity_counts_dict = {
    LABEL_NAMES[i]: all_ner_tags.count(i)
    for i in TARGET_ENTITY_IDS
}
entity_distribution = {
    'PER': entity_counts_dict['B-PER'] + entity_counts_dict['I-PER'],
    'LOC': entity_counts_dict['B-LOC'] + entity_counts_dict['I-LOC'],
    'ORG': entity_counts_dict['B-ORG'] + entity_counts_dict['I-ORG'],
    'MISC': entity_counts_dict['B-MISC'] + entity_counts_dict['I-MISC']
}
total_entities = sum(entity_distribution.values())

# Initialize W&B and Log Summary Metrics
wandb.init(project="Q1-weak-supervision-ner", name="Q1_dataset_stats")

wandb.run.summary.update({
    "num_train_samples": num_train, "num_validation_samples": num_val, "num_test_samples": num_test,
    "Total_Entities_in_Train_Set": total_entities,
    "Entity_Distribution/PER_Count": entity_distribution['PER'],
    "Entity_Distribution/LOC_Count": entity_distribution['LOC'],
    "Entity_Distribution/ORG_Count": entity_distribution['ORG'],
    "Entity_Distribution/MISC_Count": entity_distribution['MISC'],
    "Entity_Distribution/PER_Percentage": (entity_distribution['PER'] / total_entities) * 100,
    "Entity_Distribution/LOC_Percentage": (entity_distribution['LOC'] / total_entities) * 100,
})

wandb.finish()


Found cached dataset conll2003 (/Users/fysiki_mac/.cache/huggingface/datasets/eriktks___conll2003/conll2003/1.0.0/9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98)


  0%|          | 0/3 [00:00<?, ?it/s]

C100_S1_accuracy,▁▂▃▄▄▅▅▅▆▆▆▇▇▇▇█████
C100_S1_loss,█▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
C100_S1_accuracy,94.088
C100_S1_loss,0.21116
epoch,20


Entity_Distribution/LOC_Count,10025
Entity_Distribution/LOC_Percentage,29.44805
Entity_Distribution/MISC_Count,4593
Entity_Distribution/ORG_Count,8297
Entity_Distribution/PER_Count,11128
Entity_Distribution/PER_Percentage,32.68807
Total_Entities_in_Train_Set,34043
num_test_samples,3453
num_train_samples,14041
num_validation_samples,3250


In [6]:
# ====================================================================
# Q2: Final Robust Code Block (Addressing all API Errors)
# ====================================================================
# Assumes necessary imports and data (L_train, train_df, ABSTAIN) are available.

# --- 1. Evaluate LFs for Coverage (Final Robust Attempt) ---

print("\n--- Starting Q2: Snorkel Labeling Functions (Final Robust Fix) ---")

# Fix 1: Initialize LFAnalysis without 'Y' (to avoid TypeError)
from snorkel.labeling import LFAnalysis
lf_analysis = LFAnalysis(L=L_train, lfs=lfs)

# Get the list of LF names using the CORRECT property: .name
# This is the key fix for the AttributeError: '__name__'
lf_names = [lf.name for lf in lfs] 

# --- 2. Manual Metric Calculation (for 100% reliability) ---

# Re-implement the accurate metric function based on the token-level DataFrame:
def calculate_lf_metrics_manual(L_matrix, true_labels, lf_idx, lf_name):
    # This function is unchanged and robust.
    predictions = L_matrix[:, lf_idx]
    
    # Coverage calculation
    coverage = np.sum(predictions != ABSTAIN) / len(predictions)
    
    # Accuracy calculation (only on predicted samples)
    predicted_indices = predictions != ABSTAIN
    if np.sum(predicted_indices) > 0:
        correct = np.sum(predictions[predicted_indices] == true_labels[predicted_indices])
        accuracy = correct / np.sum(predicted_indices)
    else:
        accuracy = 0.0
    
    return {'coverage': coverage, 'accuracy': accuracy}

# Calculate metrics using the manually defined function
metrics_years = calculate_lf_metrics_manual(L_train, train_df['true_label'].values, 0, lf_names[0])
metrics_org = calculate_lf_metrics_manual(L_train, train_df['true_label'].values, 1, lf_names[1])


# --- 3. Log to W&B ---
import wandb
wandb.init(project="Q1-weak-supervision-ner", name="Q2_lf_evaluation_final")

# Log coverage and accuracy using the manually calculated reliable values
wandb.log({
    "lf_year_coverage": metrics_years['coverage'],
    "lf_year_accuracy": metrics_years['accuracy'],
    
    "lf_org_coverage": metrics_org['coverage'],
    "lf_org_accuracy": metrics_org['accuracy']
})

# Add the final metrics to the summary
wandb.summary['lf_year_coverage'] = metrics_years['coverage']
wandb.summary['lf_year_accuracy'] = metrics_years['accuracy']
wandb.summary['lf_org_coverage'] = metrics_org['coverage']
wandb.summary['lf_org_accuracy'] = metrics_org['accuracy']

wandb.finish()



--- Starting Q2: Snorkel Labeling Functions (Final Robust Fix) ---


lf_org_accuracy,▁
lf_org_coverage,▁
lf_year_accuracy,▁
lf_year_coverage,▁
lf_org_accuracy,0
lf_org_coverage,0.00011
lf_year_accuracy,0.00552
lf_year_coverage,0.00267


Q2 finished. LF coverage and accuracy logged using the final robust method.


In [7]:
# ====================================================================
# Q3: Implement Snorkel's Label aggregation (Majority Label Voter) (FIXED)
# ====================================================================
print("\n--- Starting Q3: Majority Label Voter ---")

# 1. Initialize the MajorityLabelVoter with the correct CARDINALITY
# This fixes the IndexError when the voter encounters labels 7 or 5.
label_model = MajorityLabelVoter(cardinality=CARDINALITY)

# 2. Generate aggregated predictions (weak labels)
preds_train = label_model.predict(L=L_train)

# Calculate the model's coverage
num_labeled = (preds_train != ABSTAIN).sum()
coverage_voter = num_labeled / len(preds_train)

# --- Logging ---
wandb.init(project="Q1-weak-supervision-ner", name="Q3_label_aggregation")

wandb.log({
    "majority_voter_labeled_samples": num_labeled,
    "majority_voter_total_samples": len(preds_train),
    "majority_voter_coverage": coverage_voter,
})

wandb.finish()
print(f"Q3 finished. Majority Voter coverage: {coverage_voter:.4f} logged.")




--- Starting Q3: Majority Label Voter ---


majority_voter_coverage,▁
majority_voter_labeled_samples,▁
majority_voter_total_samples,▁
majority_voter_coverage,0.00277
majority_voter_labeled_samples,565
majority_voter_total_samples,203621


Q3 finished. Majority Voter coverage: 0.0028 logged.


In [8]:

# ====================================================================
# Q4: Sequential Training with W&B (CIFAR 10/100) (COMPLETED)
# ====================================================================
print("\n--- Starting Q4: Sequential CIFAR Training ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS_PER_STAGE = 100 
BATCH_SIZE = 128

# Simple CNN Model 
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 32x16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)  # 64x8x8
        )
        self.fc = nn.Linear(64 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.conv_stack(x)
        x = x.view(x.size(0), -1) 
        x = self.fc(x)
        return x
        
    def update_num_classes(self, new_num_classes):
        in_features = self.fc.in_features
        self.fc = nn.Linear(in_features, new_num_classes)
        return self # Return self to allow chaining .to(device)

# Data Transformations and Loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainloader_10 = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform), 
    batch_size=BATCH_SIZE, shuffle=True
)
trainloader_100 = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform), 
    batch_size=BATCH_SIZE, shuffle=True
)

# Training Function
def train_model_stage(model, dataloader, criterion, optimizer, num_epochs, task_name, log_offset):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100 * correct / total
        
        wandb.log({
            "epoch": epoch + 1 + log_offset,
            f"{task_name}_loss": epoch_loss,
            f"{task_name}_accuracy": epoch_acc,
        })
        print(f"Epoch {epoch+1+log_offset} ({task_name}): Loss={epoch_loss:.4f}, Acc={epoch_acc:.2f}%")
    return model

# --- Experiment (a): CIFAR 100 then CIFAR 10 ---
wandb.init(project="Q4-cifar-experiments", name="CIFAR100_then_10")

# Stage 1: C100 (100 classes)
model_a = SimpleCNN(num_classes=100).to(device)
criterion_a = nn.CrossEntropyLoss()
optimizer_a = optim.Adam(model_a.parameters(), lr=0.001)

model_a = train_model_stage(model_a, trainloader_100, criterion_a, optimizer_a, 
                            num_epochs=EPOCHS_PER_STAGE, task_name="C100_S1", log_offset=0)

# Stage 2: C10 (10 classes)
model_a = model_a.update_num_classes(10).to(device) 
optimizer_a = optim.Adam(model_a.parameters(), lr=0.001) # Reset optimizer
model_a = train_model_stage(model_a, trainloader_10, criterion_a, optimizer_a, 
                            num_epochs=EPOCHS_PER_STAGE, task_name="C10_S2", log_offset=EPOCHS_PER_STAGE)

wandb.finish()


# --- Experiment (b): CIFAR 10 then CIFAR 100 ---
wandb.init(project="Q4-cifar-experiments", name="CIFAR10_then_100")

# Stage 1: C10 (10 classes)
model_b = SimpleCNN(num_classes=10).to(device)
criterion_b = nn.CrossEntropyLoss()
optimizer_b = optim.Adam(model_b.parameters(), lr=0.001)

model_b = train_model_stage(model_b, trainloader_10, criterion_b, optimizer_b, 
                            num_epochs=EPOCHS_PER_STAGE, task_name="C10_S1", log_offset=0)

# Stage 2: C100 (100 classes)
model_b = model_b.update_num_classes(100).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=0.001) # Reset optimizer
model_b = train_model_stage(model_b, trainloader_100, criterion_b, optimizer_b, 
                            num_epochs=EPOCHS_PER_STAGE, task_name="C100_S2", log_offset=EPOCHS_PER_STAGE)

wandb.finish()
print("\nQ4 finished. Two sequential runs logged to W&B.")


--- Starting Q4: Sequential CIFAR Training ---
Files already downloaded and verified
Files already downloaded and verified


Epoch 1 (C100_S1): Loss=3.4049, Acc=20.80%
Epoch 2 (C100_S1): Loss=2.6422, Acc=35.20%
Epoch 3 (C100_S1): Loss=2.2658, Acc=43.33%
Epoch 4 (C100_S1): Loss=1.9785, Acc=49.48%
Epoch 5 (C100_S1): Loss=1.7293, Acc=54.97%
Epoch 6 (C100_S1): Loss=1.5103, Acc=60.05%


KeyboardInterrupt: 